# Daily Challenge: Pinecone Serverless Reranking in Action

Reranking models score each *(query, document)* pair jointly (a **cross-encoder**) and reorder the results so that the most relevant ones come first. A typical pipeline is:
1. **fast retrieval**: a bi-encoder embedding plus a vector index returns the top-k candidates;
2. **precise reranking**: a cross-encoder re-scores only those k candidates.

We use Pinecone's hosted `bge-reranker-v2-m3` on (1) a toy Apple-company vs apple-fruit example and (2) medical notes stored in a Pinecone serverless index.

> **How to run it:** on Colab, run the authentication cell and paste your Pinecone API key (free account at pinecone.io). All Pinecone calls then go to your project.
>
> **Offline fallback:** this copy was executed in an environment without a Pinecone API key. In that case the notebook automatically uses a **local backend with the same interface**:
> - the *same* reranker model, `BAAI/bge-reranker-v2-m3`, run locally with `sentence-transformers`, with the same sigmoid scores in [0, 1];
> - an in-memory cosine index that mimics `create_index` / `upsert_from_dataframe` / `query`.
>
> All the exercise code below is unchanged and works with both backends; the cell after the client creation tells you which one is active.

## Part 1 · Load documents & execute the reranking model

### 1. Install Pinecone libraries

In [1]:
!pip install -q -U pinecone==6.0.1 pinecone-notebooks

### 2. Authenticate with Pinecone

In [2]:
import os
os.environ["USE_TF"] = "0"     # transformers: PyTorch only

if not os.environ.get("PINECONE_API_KEY"):
    try:
        from pinecone_notebooks.colab import Authenticate
        Authenticate()          # Colab pop-up: log in / paste your API key
    except ImportError as e:    # not running in Colab
        print("Pinecone Colab authentication not available here:", e)

Pinecone Colab authentication not available here: This module is meant to be used in Google Colab.


### 3. Instantiate the Pinecone client

In [3]:
from pinecone import Pinecone
api_key = os.environ.get("PINECONE_API_KEY")
USE_PINECONE = bool(api_key)

if USE_PINECONE:
    pc = Pinecone(api_key=api_key)
    print("Using Pinecone cloud. Indexes:", pc.list_indexes().names())
else:
    print("No PINECONE_API_KEY found -> using the local backend defined in the next cell.")

No PINECONE_API_KEY found -> using the local backend defined in the next cell.


In [4]:
# ---------------------------------------------------------------------------------------------
# Local backend (only used when no Pinecone API key is available).
# It reproduces the small part of the Pinecone API used in this challenge, with the same
# reranker model (BAAI/bge-reranker-v2-m3) so that scores and rankings are comparable.
# ---------------------------------------------------------------------------------------------
from types import SimpleNamespace
import numpy as np

class _Doc(dict):
    # Pinecone's reranked documents support both attribute access (doc.text) and dict access (doc["text"])
    __getattr__ = dict.get

class LocalInference:
    def __init__(self):
        self._models = {}
    def rerank(self, model, query, documents, top_n=None, rank_fields=("text",), return_documents=True, parameters=None):
        from sentence_transformers import CrossEncoder
        hf_name = {"bge-reranker-v2-m3": "BAAI/bge-reranker-v2-m3"}[model]
        if hf_name not in self._models:
            self._models[hf_name] = CrossEncoder(hf_name)          # sigmoid output in [0, 1], like Pinecone
        docs = [d if isinstance(d, dict) else {"text": d} for d in documents]
        texts = ["\n".join(str(d[f]) for f in rank_fields) for d in docs]
        scores = self._models[hf_name].predict([(query, t) for t in texts])
        order = np.argsort(-scores)[: top_n or len(docs)]
        data = [SimpleNamespace(index=int(i), score=float(scores[i]),
                                document=_Doc(docs[i]) if return_documents else None) for i in order]
        return SimpleNamespace(model=model, data=data, usage={"rerank_units": 1})

class LocalIndex:
    def __init__(self, dimension, metric):
        self.dimension, self.metric = dimension, metric
        self.ids, self.vectors, self.metadata = [], np.zeros((0, dimension), dtype=np.float32), []
    def upsert_from_dataframe(self, df, batch_size=500, show_progress=True):
        vecs = np.array(df["values"].tolist(), dtype=np.float32)
        assert vecs.shape[1] == self.dimension, f"Dimension mismatch: {vecs.shape[1]} vs {self.dimension}"
        self.ids += df["id"].tolist(); self.metadata += df["metadata"].tolist()
        self.vectors = np.vstack([self.vectors, vecs])
        return {"upserted_count": len(df)}
    def describe_index_stats(self):
        return SimpleNamespace(dimension=self.dimension, total_vector_count=len(self.ids),
                               namespaces={"": {"vector_count": len(self.ids)}}, metric=self.metric)
    def query(self, vector, top_k=10, include_metadata=True):
        q = np.array(vector, dtype=np.float32).reshape(-1)
        V = self.vectors
        scores = (V @ q) / (np.linalg.norm(V, axis=1) * np.linalg.norm(q) + 1e-12)   # cosine
        top = np.argsort(-scores)[:top_k]
        return {"matches": [{"id": self.ids[i], "score": float(scores[i]),
                             "metadata": self.metadata[i] if include_metadata else None} for i in top]}

class LocalPinecone:
    def __init__(self):
        self.inference, self._indexes = LocalInference(), {}
    def has_index(self, name): return name in self._indexes
    def delete_index(self, name): self._indexes.pop(name, None)
    def create_index(self, name, dimension, metric, spec=None): self._indexes[name] = LocalIndex(dimension, metric)
    def Index(self, name): return self._indexes[name]

if not USE_PINECONE:
    pc = LocalPinecone()
print("Backend:", "Pinecone serverless (cloud)" if USE_PINECONE else "local (same reranker model, in-memory index)")

Backend: local (same reranker model, in-memory index)


### 4. Define your query & documents

In [5]:
query = "Tell me about Apple's products"
documents = [
    "Apple is a popular fruit known for its sweetness and crisp texture.",                       # fruit
    "Apple Inc. designs the iPhone, iPad, Mac computers, Apple Watch and AirPods.",               # company products
    "Many apple varieties, such as Granny Smith and Fuji, are grown in orchards and used in pies.",  # fruit
    "Apple announced new MacBook Pro laptops powered by its own M-series chips.",                 # company
    "Eating an apple a day provides fiber and vitamin C, and may keep the doctor away.",          # fruit / health
]

### 5. Call the reranker

In [6]:
from pinecone import RerankModel
print("Pinecone rerank models:", [m.value for m in RerankModel])

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3,
    return_documents=True,
)

Pinecone rerank models: ['bge-reranker-v2-m3', 'cohere-rerank-3.5', 'pinecone-rerank-v0']


### 6. Inspect reranked results

In [7]:
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"  {i+1}. score={m.score:.4f} | id={m.document.id} | {m.document.text}")

show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products
  1. score=0.6905 | id=1 | Apple Inc. designs the iPhone, iPad, Mac computers, Apple Watch and AirPods.
  2. score=0.1743 | id=0 | Apple is a popular fruit known for its sweetness and crisp texture.
  3. score=0.0831 | id=3 | Apple announced new MacBook Pro laptops powered by its own M-series chips.


In [8]:
# Different top_n values (and the full ranking) to see how the ordering behaves
full = pc.inference.rerank(model="bge-reranker-v2-m3", query=query,
                           documents=[{"id": str(i), "text": d} for i, d in enumerate(documents)],
                           top_n=len(documents), return_documents=True)
show_reranked_results(query + "   [top_n = all]", full.data)
print()
q2 = "Which fruit is good for health?"
show_reranked_results(q2, pc.inference.rerank(model="bge-reranker-v2-m3", query=q2,
                      documents=[{"id": str(i), "text": d} for i, d in enumerate(documents)], top_n=3).data)

Query: Tell me about Apple's products   [top_n = all]
  1. score=0.6905 | id=1 | Apple Inc. designs the iPhone, iPad, Mac computers, Apple Watch and AirPods.
  2. score=0.1743 | id=0 | Apple is a popular fruit known for its sweetness and crisp texture.
  3. score=0.0831 | id=3 | Apple announced new MacBook Pro laptops powered by its own M-series chips.
  4. score=0.0114 | id=2 | Many apple varieties, such as Granny Smith and Fuji, are grown in orchards and used in pies.
  5. score=0.0017 | id=4 | Eating an apple a day provides fiber and vitamin C, and may keep the doctor away.



Query: Which fruit is good for health?
  1. score=0.3739 | id=4 | Eating an apple a day provides fiber and vitamin C, and may keep the doctor away.
  2. score=0.0770 | id=0 | Apple is a popular fruit known for its sweetness and crisp texture.
  3. score=0.0615 | id=2 | Many apple varieties, such as Granny Smith and Fuji, are grown in orchards and used in pies.


**Part 1 analysis.** `pc.inference.rerank` returns an object whose **`.data`** list holds one entry per document, ordered by decreasing **`.score`**, the relevance probability in [0, 1] obtained from a sigmoid. Each entry gives the document through **`.document`** (`.id`, `.text`).

- **The reranker separates the two meanings of "Apple".** *"Apple Inc. designs the iPhone, iPad, Mac…"* is clearly first (**0.69**), and the two clearly fruit-related documents (orchard varieties 0.011, "an apple a day" 0.002) are last.
- **It is not perfect.** The short sentence *"Apple is a popular fruit…"* (0.17) comes **before** the MacBook announcement (0.08), which is about an Apple product without saying the word "product". Cross-encoders reward lexical and semantic overlap with the query ("Apple" at the start, a short definition-like sentence), so the scores should be read as a ranking, not as calibrated probabilities.
- **`top_n` only truncates.** With `top_n=3` vs `top_n=5` the order of the first three documents is identical: reranking scores every document and then keeps the best `top_n`.
- **The query decides the order.** With *"Which fruit is good for health?"* the fruit/health documents move to the top (0.37 for "an apple a day…") and the company documents drop out of the top 3.

## Part 2 · Set up a serverless index for medical notes

### 1. Install data & model libraries

In [9]:
!pip install -q pandas torch transformers

### 2. Import modules & define environment settings

In [10]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings (these are defaults that work for most users)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'
print(spec)

ServerlessSpec(cloud='aws', region='us-east-1')


### 3. Create or recreate the index

In [11]:
# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,     # all-MiniLM-L6-v2 outputs 384-d vectors
    metric='cosine',   # cosine similarity for text embeddings
    spec=spec
)
print(f"Index '{index_name}' created")

Index 'medical-notes-index' created


## Part 3 · Load the sample data

In [12]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/refs/heads/main/Week%208/W8D2/data.json"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

In [13]:
# Show head of the DataFrame
print("Data shape:", df.shape)
print("Embedding size:", len(df["values"][0]))
df.head()

Data shape: (100, 3)
Embedding size: 384


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [14]:
from collections import Counter
print("Metadata fields:", dict(Counter(k for m in df["metadata"] for k in m).most_common()))
pd.DataFrame(df["metadata"].tolist(), index=df["id"]).head(12)

Metadata fields: {'symptoms': 36, 'condition': 36, 'treatment': 28, 'advice': 17, 'medications': 11, 'diagnosis': 9, 'tests': 8, 'referral': 8, 'vital_signs': 8, 'surgery': 8, 'blood_pressure': 8, 'age': 8, 'vaccines': 8, 'development': 8, 'vitals': 8, 'recovery': 7, 'blood_sugar': 5, 'HbA1c': 4, 'exam': 2, 'cholesterol': 2, 'deficiency': 2, 'BMI': 2, 'preventive_care': 1, 'procedure': 1}


,advice,symptoms,tests,HbA1c,condition,medications,diagnosis,treatment,referral,vital_signs,...,age,vaccines,development,exam,vitals,blood_sugar,recovery,cholesterol,deficiency,BMI
id,,,,,,,,,,,,,,,,,,,,,
P011,"rest, hydrate",headache,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P001,NaN,chest pain,"EKG, stress test",NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P002,NaN,NaN,NaN,7.2,diabetes,metformin,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P003,NaN,"cough, wheezing",NaN,NaN,NaN,NaN,bronchitis,antibiotics,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P004,NaN,NaN,NaN,NaN,suspicious mole,NaN,NaN,NaN,dermatology,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,normal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P006,NaN,"anxiety, insomnia",NaN,NaN,NaN,NaN,NaN,"SSRI, CBT",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P007,NaN,"pain, swelling",NaN,NaN,NaN,NaN,NaN,physical therapy,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
P008,NaN,NaN,NaN,NaN,appendicitis,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Part 4 · Upsert data into the index

In [15]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df)

{'upserted_count': 100}

In [16]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

Vector count:  100
Index ready!


namespace(dimension=384,
          total_vector_count=100,
          namespaces={'': {'vector_count': 100}},
          metric='cosine')

## Part 5 · Query & embedding function
`last_hidden_state[0]` has shape `(sequence_length, 384)`: one vector per token. Averaging over **dim=0** (the sequence dimension) gives one 384-d sentence vector (mean pooling), in the same space as the stored notes.

We load the tokenizer and model **once**, outside the function. The skeleton reloads them at every call, which works but is slow.

In [17]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(input_question):
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    embedding = model_output.last_hidden_state[0].mean(dim=0)    # average over the tokens
    return embedding

print(get_embedding("patient has chest pain").shape)

torch.Size([384])


In [18]:
# Build a query to search
question = "what if my patient has a knee injury"
query = get_embedding(question).tolist()

# Get results
# Pinecone expects a flat list of floats (the skeleton's `[query]` would be a nested list)
results = index.query(vector=query, top_k=10, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

## Part 6 · Display & rerank clinical notes

In [19]:
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

Question: 'what if my patient has a knee injury'

Results:
   1. ID: P007
 Score: 0.6318659782409668
 Metadata: {'surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy'}

   2. ID: P028
 Score: 0.6090426445007324
 Metadata: {'referral': 'orthopedics', 'condition': 'knee pain'}

   3. ID: P068
 Score: 0.4873522222042084
 Metadata: {'condition': 'broken arm', 'treatment': 'cast'}

   4. ID: P031
 Score: 0.4775209128856659
 Metadata: {'surgery': 'hip replacement', 'recovery': 'mobility improvement'}

   5. ID: P020
 Score: 0.46115222573280334
 Metadata: {'tests': 'X-ray', 'condition': 'sprained ankle'}

   6. ID: P059
 Score: 0.43229106068611145
 Metadata: {'symptoms': 'joint pain', 'treatment': 'NSAIDs, rest'}

   7. ID: P080
 Score: 0.4281395971775055
 Metadata: {'condition': 'concussion', 'treatment': 'rest, follow-up'}

   8. ID: P047
 Score: 0.4037027359008789
 Metadata: {'symptoms': 'back pain', 'treatment': 'physical therapy'}

   9. ID: P095
 S

In [20]:
# Create documents with concatenated metadata field as "reranking_field" field
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]
transformed_documents[:3]

[{'id': 'P007',
  'reranking_field': 'surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy'},
 {'id': 'P028',
  'reranking_field': 'referral: orthopedics; condition: knee pain'},
 {'id': 'P068', 'reranking_field': 'condition: broken arm; treatment: cast'}]

In [21]:
# Define a more specific query for reranking
refined_query = "what if my patient has a knee injury and needs surgery?"

# Perform reranking based on the query and specified field
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

In [22]:
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)

Question: 'what if my patient has a knee injury and needs surgery?'

Reranked Results:
   1. ID: P007
 Score: 0.11052493005990982
 Reranking Field: surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy

   2. ID: P031
 Score: 0.008204018697142601
 Reranking Field: surgery: hip replacement; recovery: mobility improvement

   3. ID: P028
 Score: 0.004569997079670429
 Reranking Field: referral: orthopedics; condition: knee pain



### Comparison: vector search vs reranking (several clinical questions)

In [23]:
def compare(question, refined_query=None, top_k=10, top_n=3):
    refined_query = refined_query or question
    q = get_embedding(question).tolist()
    res = index.query(vector=q, top_k=top_k, include_metadata=True)
    matches = sorted(res["matches"], key=lambda x: x["score"], reverse=True)
    docs = [{"id": m["id"], "reranking_field": "; ".join(f"{k}: {v}" for k, v in m["metadata"].items())} for m in matches]
    rr = pc.inference.rerank(model="bge-reranker-v2-m3", query=refined_query, documents=docs,
                             rank_fields=["reranking_field"], top_n=top_n, return_documents=True)
    vec_top = pd.DataFrame({"vector rank": range(1, top_n + 1),
                            "id": [m["id"] for m in matches[:top_n]],
                            "cosine": [round(m["score"], 3) for m in matches[:top_n]],
                            "note": [d["reranking_field"] for d in docs[:top_n]]})
    rr_top = pd.DataFrame({"rerank rank": range(1, top_n + 1),
                           "id": [r.document.id for r in rr.data],
                           "rerank score": [round(r.score, 4) for r in rr.data],
                           "was vector rank": [[m["id"] for m in matches].index(r.document.id) + 1 for r in rr.data],
                           "note": [r.document.reranking_field for r in rr.data]})
    print(f"Vector search: {question!r}"); display(vec_top)
    print(f"Reranked with: {refined_query!r}"); display(rr_top)

compare("patient has chest pain", "patient with chest pain: which cardiac tests should be ordered?")
compare("diabetes", "diabetes treatment plan with medication")
compare("child vaccination", "which vaccines did the 5 year old child receive?")

Vector search: 'patient has chest pain'


,vector rank,id,cosine,note
0,1,P001,0.734,"tests: EKG, stress test; symptoms: chest pain"
1,2,P016,0.484,referral: cardiology; condition: heart murmur
2,3,P0100,0.447,"advice: over-the-counter pain relief, stretchi..."


Reranked with: 'patient with chest pain: which cardiac tests should be ordered?'


,rerank rank,id,rerank score,was vector rank,note
0,1,P001,0.8379,1,"tests: EKG, stress test; symptoms: chest pain"
1,2,P016,0.0006,2,referral: cardiology; condition: heart murmur
2,3,P0100,0.0002,3,"advice: over-the-counter pain relief, stretchi..."


Vector search: 'diabetes'


...

,rerank rank,id,rerank score,was vector rank,note
0,1,P014,0.2466,1,condition: diabetes; blood_sugar: 8.0; medicat...
1,2,P074,0.1302,4,condition: diabetes; blood_sugar: 7.2; medicat...
2,3,P050,0.1276,3,condition: diabetes; blood_sugar: 7.0; medicat...


Vector search: 'child vaccination'


,vector rank,id,cosine,note
0,1,P058,0.644,age: 2; vaccines: DTaP; development: normal
1,2,P034,0.633,age: 4; vaccines: varicella; development: normal
2,3,P082,0.633,age: 4; vaccines: varicella; development: normal


Reranked with: 'which vaccines did the 5 year old child receive?'


,rerank rank,id,rerank score,was vector rank,note
0,1,P010,0.9444,7,"age: 5; vaccines: MMR, DTaP; development: normal"
1,2,P022,0.2764,4,"age: 6; vaccines: polio, hepatitis; developmen..."
2,3,P082,0.2298,3,age: 4; vaccines: varicella; development: normal


### Part 2–6 analysis: why rerank?
**Vector search** (bi-encoder `all-MiniLM-L6-v2` + cosine index) encodes the query and each note *separately* and compares the two vectors. It is fast and scales to millions of vectors, but it is coarse: for *"what if my patient has a knee injury"* the top 10 includes a broken arm, a sprained ankle, a concussion and back pain (cosine 0.40–0.49), all close to "injury" in general.

**Reranking** (cross-encoder `bge-reranker-v2-m3`) reads the query *and* the note together, so it can check whether the note actually answers the question.

| refined query | effect of reranking |
|---|---|
| *knee injury and needs surgery?* | **P007 (knee arthroscopy)** stays first, with a large margin (0.11 vs 0.008). **P031 hip replacement** (vector rank 4) moves up to 2nd because it is a *surgery*, while "knee pain → orthopedics" drops to 3rd. The arm/ankle/concussion notes disappear from the top 3. |
| *chest pain: which cardiac tests should be ordered?* | Same order as vector search, but the reranker is far more decisive: **P001 (EKG, stress test) 0.84** vs 0.0006 for the next one. Vector search gave 0.73 vs 0.48. |
| *diabetes treatment plan with medication* | **P074** (vector rank 4) comes up to 2nd: notes that list *medications* are preferred over the dietary-advice note P026 (vector rank 2). |
| *which vaccines did the 5 year old child receive?* | The best example: **P010 (age 5, MMR, DTaP), only 7th in vector search, becomes 1st with 0.94**. The bi-encoder cannot compare the number "5" with `age: 5`, but the cross-encoder can. |

**Takeaways.**
- Retrieval → rerank is a **two-stage pipeline**: a cheap vector search returns top-k candidates (high recall), then an expensive but precise cross-encoder reorders them (high precision). Reranking all 100 notes, let alone millions, would be too slow; reranking 10 candidates is cheap.
- **The reranker can only reorder what retrieval returned.** If the right note is not in the top-k, reranking cannot recover it, so k must be large enough (here 10).
- **The text given to the reranker matters.** Here it is the metadata flattened into `"key: value; …"`. Clear field names (`surgery`, `vaccines`, `age`) help the model match the question.
- **Scores are very peaked** (0.84 vs 0.0006), which makes them convenient for a confidence threshold, e.g. showing a clinician only notes above a given score.
- In a clinical setting this ordering matters: the most pertinent note (the actual knee surgery, the right child's vaccination record) appears first instead of a merely similar one.

*Executed with the local backend, using the same `bge-reranker-v2-m3` model as Pinecone's hosted reranker. With a Pinecone API key the same cells run against Pinecone serverless; the scores may differ slightly (Pinecone may truncate or normalise inputs differently), but the ranking logic is the same.*

### Clean up (optional)

In [24]:
# Delete the index to save resources
pc.delete_index(name=index_name)
print("Index deleted:", not pc.has_index(name=index_name))

Index deleted: True
